In [4]:
# Baseline Sales Forecasting Model with Versioning
# This notebook implements a versioned baseline for the DATATHON 2026 sales forecasting task.

import sys
import os

sys.path.append("../src")

from data.load_data import load_sales_data
from data.preprocess import preprocess_sales_data
from features.build_features import build_sales_features
from models.train import train_baseline_model
from models.predict import predict_baseline, create_submission
from models.evaluate import evaluate_model

# Import versioning utilities
from utils.versioning import VersionManager, get_latest_model_version
from utils.reproducibility import load_model_metadata, load_submission_metadata

import pandas as pd
import numpy as np

In [5]:
# Load and preprocess training data
train_df = load_sales_data(train=True)
train_df = preprocess_sales_data(train_df, train=True)
train_df = build_sales_features(train_df)

# Features for training
X_train = train_df[["days_since_start"]]
y_revenue = train_df["Revenue"]
y_cogs = train_df["COGS"]

print("Training data shape:", train_df.shape)
print(train_df.head())

Training data shape: (3833, 7)
        Date     Revenue        COGS  days_since_start  day_of_week  month  \
0 2012-07-04  5123547.94  3982991.19                 0            2      7   
1 2012-07-05  2751773.45  2150580.23                 1            3      7   
2 2012-07-06  3054029.42  2517632.84                 2            4      7   
3 2012-07-07  2667930.94  2108246.62                 3            5      7   
4 2012-07-08  2360851.90  1808622.79                 4            6      7   

   year  
0  2012  
1  2012  
2  2012  
3  2012  
4  2012  


In [6]:
# Train baseline models with versioning
# Define hyperparameters for reproducibility
hyperparameters = {
    "model_type": "LinearRegression",
    "fit_intercept": True,
    "features_used": ["days_since_start"],
    "features_count": X_train.shape[1],
    "preprocessing": "build_sales_features",
    "notebook": "modeling.ipynb",
    "experiment": "Baseline model with days_since_start feature",
    "comment": "First baseline model in modeling notebook",
}

# Train with versioning (creates model_v0, model_v1, etc.)
revenue_model, cogs_model, version_num, model_dir = train_baseline_model(
    X_train,
    y_revenue,
    y_cogs,
    hyperparameters=hyperparameters,
    create_version=True, 
)

print(f"✓ Models trained and saved as version {version_num}")
print(f"✓ Model location: {model_dir}")
print(f"✓ Hyperparameters logged: {hyperparameters['comment']}")

# Store version for later use
MODEL_VERSION = version_num

2026-04-20 00:50:28,349 - version_v0 - INFO - Hyperparameters: {
  "model_type": "LinearRegression",
  "fit_intercept": true,
  "features_used": [
    "days_since_start"
  ],
  "features_count": 1,
  "preprocessing": "build_sales_features",
  "notebook": "modeling.ipynb",
  "experiment": "Baseline model with days_since_start feature",
  "comment": "First baseline model in modeling notebook"
}
2026-04-20 00:50:28,350 - version_v0 - INFO - Starting training for model version 0
2026-04-20 00:50:28,356 - version_v0 - INFO - Revenue model saved to ../outputs/models/model_v0\revenue_model.pkl
2026-04-20 00:50:28,357 - version_v0 - INFO - COGS model saved to ../outputs/models/model_v0\cogs_model.pkl
2026-04-20 00:50:28,361 - version_v0 - INFO - training_revenue_r2_score: 0.08750556726178826
2026-04-20 00:50:28,362 - version_v0 - INFO - training_cogs_r2_score: 0.0804331311775538
2026-04-20 00:50:28,362 - version_v0 - INFO - training_n_features: 1
2026-04-20 00:50:28,362 - version_v0 - INFO - t

✓ Models trained and saved as version 0
✓ Model location: ../outputs/models/model_v0
✓ Hyperparameters logged: First baseline model in modeling notebook


In [7]:
# Evaluate on training data
pred_revenue_train = revenue_model.predict(X_train)
pred_cogs_train = cogs_model.predict(X_train)

print(f"📊 Evaluating Model v{MODEL_VERSION}")
evaluate_model(y_revenue, pred_revenue_train, "Revenue Model")
evaluate_model(y_cogs, pred_cogs_train, "COGS Model")

# Load and display metadata
model_meta = load_model_metadata("../outputs/models/", MODEL_VERSION)
print(f"\n📋 Model v{MODEL_VERSION} Metadata:")
print(f"   Features used: {model_meta['hyperparameters']['features_used']}")
print(
    f"   Training R² Revenue: {model_meta['training_metrics']['training_revenue_r2_score']:.4f}"
)
print(
    f"   Training R² COGS: {model_meta['training_metrics']['training_cogs_r2_score']:.4f}"
)
print(f"   Created: {model_meta['timestamp'][:19]}")

📊 Evaluating Model v0
Revenue Model - RMSE: 2507040.33, MAE: 1843133.44, R²: 0.0875
COGS Model - RMSE: 2128367.64, MAE: 1565849.11, R²: 0.0804

📋 Model v0 Metadata:
   Features used: ['days_since_start']
   Training R² Revenue: 0.0875
   Training R² COGS: 0.0804
   Created: 2026-04-20T00:50:28


In [8]:
# Load and preprocess test data
test_df = load_sales_data(train=False)
test_df = preprocess_sales_data(test_df, train=False)
test_df = build_sales_features(test_df)

# Align test features with train
X_test = test_df[["days_since_start"]]

print("Test data shape:", test_df.shape)
print(test_df.head())

Test data shape: (548, 5)
        Date  days_since_start  day_of_week  month  year
0 2023-01-01                 0            6      1  2023
1 2023-01-02                 1            0      1  2023
2 2023-01-03                 2            1      1  2023
3 2023-01-04                 3            2      1  2023
4 2023-01-05                 4            3      1  2023


In [9]:
# Predict on test data using versioned model
pred_revenue, pred_cogs, model_v_used, model_path_used = predict_baseline(
    X_test,
    model_version=MODEL_VERSION,  # Use the model we just trained
)

print(f"✓ Predictions made using model v{model_v_used}")
print(f"   Model path: {model_path_used}")

# Create versioned submission
submission_path, submission_v = create_submission(
    test_df,
    pred_revenue,
    pred_cogs,
    model_version=model_v_used,
    notes=f"Baseline submission from modeling.ipynb using model v{model_v_used}",
    submission_version=None,  # Auto-increment version
)

print(f"✓ Versioned submission v{submission_v} created!")
print(f"   File: {submission_path}")

# Display submission statistics
submission_meta = load_submission_metadata("../outputs/submissions/", submission_v)
print(f"\n📋 Submission v{submission_v} Statistics:")
print(f"   Source model: v{submission_meta['source_model_version']}")
print(f"   Rows: {submission_meta['metrics']['num_rows']}")
print(f"   Revenue mean: {submission_meta['metrics']['revenue_mean']:.2f}")
print(f"   COGS mean: {submission_meta['metrics']['cogs_mean']:.2f}")
print(f"   Created: {submission_meta['timestamp'][:19]}")

# Show all available versions
print(f"\n📦 Available Versions:")
model_versions = VersionManager.list_versions("../outputs/models/", "model", "")
submission_versions = VersionManager.list_versions(
    "../outputs/submissions/", "submission", ".csv"
)
print(f"   Models: {model_versions}")
print(f"   Submissions: {submission_versions}")

print(f"\n🎯 Ready to submit: {submission_path}")

✓ Predictions made using model v0
   Model path: ../outputs/models/model_v0
Submission v0 saved to ../outputs/submissions/submission_v0.csv
Submission metadata saved to ../outputs/submissions\submission_v0_metadata.json
✓ Versioned submission v0 created!
   File: ../outputs/submissions/submission_v0.csv

📋 Submission v0 Statistics:
   Source model: v0
   Rows: 548
   Revenue mean: 5439031.79
   COGS mean: 4629527.71
   Created: 2026-04-20T00:50:28

📦 Available Versions:
   Models: [0]
   Submissions: [0]

🎯 Ready to submit: ../outputs/submissions/submission_v0.csv


In [10]:
# ============================================================================
# HƯỚNG DẪN KIỂM TRA VÀ QUẢN LÝ VERSIONS
# ============================================================================

print("🔍 HƯỚNG DẪN KIỂM TRA VERSIONS:")
print()
print("1. Từ Command Line (khuyên dùng):")
print("   python version_inspector.py --list           # Xem tất cả versions")
print("   python version_inspector.py --model 0        # Chi tiết model v0")
print("   python version_inspector.py --submission 0   # Chi tiết submission v0")
print("   python version_inspector.py --summary        # Tổng quan")
print()
print("2. Từ Python (trong notebook):")
print("   from utils.reproducibility import load_model_metadata")
print("   meta = load_model_metadata('../outputs/models/', 0)")
print("   print(meta['hyperparameters'])")
print()
print("3. Chạy lại notebook này sẽ tạo version mới tự động!")
print("   - Model: model_v1, model_v2, ...")
print("   - Submission: submission_v1, submission_v2, ...")
print()
print("4. Để dùng model cũ cho submission mới:")
print("   pred_rev, pred_cogs, _, _ = predict_baseline(X_test, model_version=0)")
print("   create_submission(test_df, pred_rev, pred_cogs, model_version=0)")
print()
print("🎯 Mỗi lần chạy notebook = 1 version mới với đầy đủ metadata!")


🔍 HƯỚNG DẪN KIỂM TRA VERSIONS:

1. Từ Command Line (khuyên dùng):
   python version_inspector.py --list           # Xem tất cả versions
   python version_inspector.py --model 0        # Chi tiết model v0
   python version_inspector.py --submission 0   # Chi tiết submission v0
   python version_inspector.py --summary        # Tổng quan

2. Từ Python (trong notebook):
   from utils.reproducibility import load_model_metadata
   meta = load_model_metadata('../outputs/models/', 0)
   print(meta['hyperparameters'])

3. Chạy lại notebook này sẽ tạo version mới tự động!
   - Model: model_v1, model_v2, ...
   - Submission: submission_v1, submission_v2, ...

4. Để dùng model cũ cho submission mới:
   pred_rev, pred_cogs, _, _ = predict_baseline(X_test, model_version=0)
   create_submission(test_df, pred_rev, pred_cogs, model_version=0)

🎯 Mỗi lần chạy notebook = 1 version mới với đầy đủ metadata!


In [13]:
path = r"C:\DATATHON_2026_DataNiga\outputs\submissions\submission_v0.csv"
res = pd.read_csv(path)
res.shape

(548, 3)